# Ducky dataset on a non-raster scanThe same ducky simulation as `ducky-dataset.ipynb`, with the probe positions on a hexagonallattice rather than a grid. It produces the dataset used by the last section of`direct_ptycho_ungridded_scans.ipynb`, where the scan is non-raster by construction ratherthan by decimating a raster one.Two things differ from the raster notebook. `abtem.GridScan` becomes `abtem.CustomScan`,which takes arbitrary `(N, 2)` positions, and the measurement comes back as `(N, 512, 512)`rather than `(scan_x, scan_y, ...)`, so the crop and bin step is a single loop over positions.This runs on CPU in roughly 45 minutes at the settings below. Set `device="gpu"` on thepotential if a GPU is available.

In [ ]:
import numpy as np
import h5py
import ase
import abtem
import dask
import matplotlib.pyplot as plt

dask.config.set({"num_workers": 1})

DEVICE = "cpu"

## Structure and potentialIdentical to the raster notebook, so the two datasets are of the same object.

In [ ]:
with h5py.File("../../../data/ducky_coords.mat", "r") as f:
    atoms = f["atoms"][:].T

ducky_atoms = ase.Atoms(
    numbers=atoms[:, -1],
    positions=atoms[:, :3],
    cell=np.array([160, 160, atoms[:, 2].max() + atoms[:, 2].min()]),
)

del ducky_atoms[(ducky_atoms.positions[:, 0] < 20) | (ducky_atoms.positions[:, 0] > 180)]
del ducky_atoms[(ducky_atoms.positions[:, 1] < 20) | (ducky_atoms.positions[:, 1] > 180)]
ducky_atoms.translate([-20, -20, 0])

potential = abtem.Potential(
    ducky_atoms,
    gpts=(1600, 1600),
    slice_thickness=0.83875,
    device=DEVICE,
).build()

probe = abtem.Probe(
    energy=80e3,
    semiangle_cutoff=20,
    defocus=500,
).match_grid(potential)

pixelated_detector = abtem.PixelatedDetector(max_angle=None)

print(f"{len(ducky_atoms)} atoms, {len(potential)} slices")

## Hexagonal scan positionsRows are spaced `spacing * sqrt(3) / 2` apart and alternate rows are offset by half aspacing. At 4.5 A nearest-neighbour spacing this gives about 1000 positions over the same148 A field the raster dataset covers, so the two are comparable in sampling density.

In [ ]:
def hexagonal_positions(extent, spacing, origin=0.0):
    """(N, 2) positions on a hexagonal lattice, in Angstrom."""
    row_pitch = spacing * np.sqrt(3) / 2
    num_rows = int(np.floor((extent - 2 * origin) / row_pitch)) + 1

    positions = []
    for row in range(num_rows):
        y = origin + row * row_pitch
        x0 = origin + (spacing / 2 if row % 2 else 0.0)
        num_cols = int(np.floor((extent - origin - x0) / spacing)) + 1
        positions.extend((y, x0 + col * spacing) for col in range(num_cols))
    return np.array(positions)


SPACING = 4.5
positions = hexagonal_positions(148.0, SPACING, origin=6.0)
custom_scan = abtem.CustomScan(positions)

print(f"{len(positions)} positions at {SPACING} A spacing")

In [ ]:
fig, ax = abtem.show_atoms(ducky_atoms)
ax.scatter(positions[:, 1], positions[:, 0], s=1.5, c="red")
ax.set_title(f"{len(positions)} hexagonal probe positions");

## ScanThe full `(N, 1600, 1600)` measurement would be about 11 GB, so we run in batches and cropand bin each one down to 200 x 200 before moving on. The crop and bin factor match the rasternotebook, giving the same 0.025 A^-1 detector sampling.

In [ ]:
BIN_FACTOR = 4
NUM_DETECTOR = 200
BATCH = 64

crop = (potential.gpts[0] - NUM_DETECTOR * BIN_FACTOR) // 2
array = np.zeros((len(positions), NUM_DETECTOR, NUM_DETECTOR), dtype=np.float32)

for start in range(0, len(positions), BATCH):
    stop = min(start + BATCH, len(positions))
    print(f"{stop}/{len(positions)}", end="\r")

    measurement = probe.scan(
        potential=potential,
        scan=abtem.CustomScan(positions[start:stop]),
        detectors=pixelated_detector,
    ).compute()

    patterns = np.asarray(measurement.array)[:, crop:-crop, crop:-crop]
    array[start:stop] = patterns.reshape(
        len(patterns), NUM_DETECTOR, BIN_FACTOR, NUM_DETECTOR, BIN_FACTOR
    ).sum((2, 4))
    del measurement, patterns

array_sums = array.sum((-2, -1))
print(f"\nSmallest array sum (should be close to 1): {array_sums.min()}")

## Dose and exportA hexagonal lattice covers `spacing**2 * sqrt(3) / 2` per position, which is what convertsthe areal dose into a per-pattern electron count. Using the same 5e4 e/A^2 as the rasterdataset keeps the noise comparable between the two.

In [ ]:
DOSE = 5e4  # e/A^2
area_per_position = SPACING**2 * np.sqrt(3) / 2
dose_per_pattern = DOSE * area_per_position

print(f"{area_per_position:.2f} A^2 per position, {dose_per_pattern:.3g} e per pattern")

np.random.seed(2025)
noisy_array = np.zeros_like(array)
for i in range(len(array)):
    pattern = array[i] / array[i].sum() * dose_per_pattern
    noisy_array[i] = np.random.poisson(pattern)

In [ ]:
from quantem.core.visualization import show_2d

show_2d(
    [array[0], noisy_array[0]],
    title=["noiseless", f"{DOSE:.0e} e/A^2"],
    power=0.5,
    cmap="turbo",
);

The patterns save as a `Dataset3d` and the positions as a `Dataset2d` in Angstrom, which isthe pair `from_dataset3d` accepts on either reconstruction class.

In [ ]:
import quantem as em

patterns_dataset = em.core.datastructures.Dataset3d.from_array(
    noisy_array,
    sampling=[1.0, 0.025, 0.025],
    units=["index", "A^-1", "A^-1"],
)

positions_dataset = em.core.datastructures.Dataset2d.from_array(
    positions,
    sampling=[1.0, 1.0],
    units=["A", "A"],
)

name = f"ducky_20mrad_500A-df_{SPACING}A-hex_{DOSE:.0e}-dose"
patterns_dataset.save(f"../../../data/{name}.zip", mode="o")
positions_dataset.save(f"../../../data/{name}_positions.zip", mode="o")
print(name)